In [ ]:
# Set parent as root and import config
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))
import configs.simulation_config as cfg

import pandas as pd
import geopandas as gpd
import osmnx as ox
import matplotlib.pyplot as plt
import numpy as np
from joblib import Parallel, delayed

Flood snapshot

In [ ]:
def ingest_flood_snapshot(path, timestamp):
    """
    Read raw CSV flood file and save as GeoParquet.
    Returns metadata row.
    """
    df = pd.read_csv(path)

    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df.x, df.y),
        crs="EPSG:4326",
    )

    ts = pd.to_datetime(timestamp)
    label = ts.strftime("%Y%m%d_%H%M")

    return {
        "label": label,
        "timestamp": ts,
        "gdf": gdf,
    }

In [ ]:
snapshots = [ingest_flood_snapshot(**snap) for snap in cfg.RAW_FLOOD_CONFIG]

graph = ox.load_graphml("../data/processed/hatyai_graph_with_dest.graphml")
nodes, edges = ox.graph_to_gdfs(graph)

edges = edges.reset_index()
nodes_proj = nodes.to_crs(epsg=32647)

nodes_with_flood = nodes_proj.drop(columns="geometry").copy()

for snap in snapshots:
    label = snap["label"]
    distance_col = f"flood_distance_{label}_m"

    right_proj = snap["gdf"][['gridcode', 'geometry']].to_crs(epsg=32647)

    joined = gpd.sjoin_nearest(
        nodes_proj,
        right_proj,
        how="left",
        distance_col=distance_col,
    ).rename(columns={"gridcode": f"flood_level_{label}"})

    joined = (
        joined.sort_values(distance_col)
        .groupby(level=0)
        .first()
        .reindex(nodes_with_flood.index)
    )

    nodes_with_flood[f"flood_level_{label}"] = joined[f"flood_level_{label}"].to_numpy()
    nodes_with_flood[distance_col] = joined[distance_col].to_numpy()

edges_with_flood = edges.copy()

for snap in snapshots:
    label = snap["label"]

    node_levels = nodes_with_flood.groupby("osmid")[f"flood_level_{label}"].max()

    u_level = edges_with_flood["u"].map(node_levels)
    v_level = edges_with_flood["v"].map(node_levels)

    edges_with_flood[f"flood_level_{label}"] = pd.concat(
        [u_level, v_level], axis=1
    ).max(axis=1)

nodes_with_flood.to_csv("../data/processed/hatyai_nodes_flood_multi.csv", index=False)

print("Flood snapshots attached (scalable):")
print("Snapshot labels:", [s["label"] for s in snapshots])
print(nodes_with_flood.filter(regex="flood_level_.*").head())
print(edges_with_flood.filter(regex="flood_level_.*").head())


Flood visualize

In [ ]:
# OPTIONAL Quick visuals: flood levels, deltas, and map
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
depth_t1 = nodes_with_flood["flood_level_20251121_0600"].map(cfg.LEVEL_TO_DEPTH)
depth_t2 = nodes_with_flood["flood_level_20251124_2200"].map(cfg.LEVEL_TO_DEPTH)
delta_depth = depth_t2 - depth_t1

depth_t1.hist(ax=axes[0], bins=30)
axes[0].set_title("Node flood depth s0 (m)")
depth_t2.hist(ax=axes[1], bins=30)
axes[1].set_title("Node flood depth s1 (m)")
delta_depth.hist(ax=axes[2], bins=30)
axes[2].set_title("Node flood depth delta (m)")
for ax in axes:
    ax.set_xlabel("meters")
plt.tight_layout()
plt.show()


Convert snapshot flood into continuous time flood levels.

In [ ]:
snapshot_labels = [s["label"] for s in snapshots]
snapshot_times = [pd.to_datetime(s["timestamp"]) for s in snapshots]
snapshot_order = np.argsort(snapshot_times)
snapshot_labels = [snapshot_labels[i] for i in snapshot_order]
snapshot_times = [snapshot_times[i] for i in snapshot_order]

# Use snapshot range unless you want to override manually
# t_start = pd.Timestamp("2025-11-24 22:00:00")
# t_end = pd.Timestamp("2025-11-26 18:00:00")
t_start = snapshot_times[0]
t_end = snapshot_times[-1]
time_index = pd.date_range(start=t_start, end=t_end, freq=cfg.TIME_STEP_FREQ)
# store for later use
timeline_df = pd.DataFrame({
    "step_id": range(len(time_index)),
    "timestamp": time_index
})
timeline_df.to_parquet(
    "../data/processed/flood_simulation_timeline.parquet",
    index=False
)

# Incremental flood update using interpolation across arbitrary snapshots
depth_to_level = {v: k for k, v in cfg.LEVEL_TO_DEPTH.items()}
depths_sorted = np.array(sorted(depth_to_level.keys()))

# Build depth matrix per edge for each snapshot
edge_depth_matrix = []
for lbl in snapshot_labels:
    edge_depth_matrix.append(
        edges_with_flood[f"flood_level_{lbl}"].map(cfg.LEVEL_TO_DEPTH).fillna(depths_sorted.min()).to_numpy()
    )
edge_depth_matrix = np.vstack(edge_depth_matrix).T  # shape: (n_edges, n_snapshots)

snapshot_hours = np.array([(ts - t_start).total_seconds() / 3600.0 for ts in snapshot_times])

# Build all interpolated flood columns in parallel (threads) to reduce wall-clock
time_hours = np.array([(ts - t_start).total_seconds() / 3600.0 for ts in time_index], dtype=float)


def interpolate_edge_depth(row):
    depth_series = np.interp(time_hours, snapshot_hours, row, left=row[0], right=row[-1])
    idx = np.abs(depth_series[:, None] - depths_sorted).argmin(axis=1)
    return depths_sorted[idx]


depth_matrix_full = Parallel(n_jobs=cfg.N_JOBS, prefer="threads")(
    delayed(interpolate_edge_depth)(row) for row in edge_depth_matrix
)
depth_matrix_full = np.vstack(depth_matrix_full)

# Map depth values back to discrete flood levels
depth_to_level_vec = np.vectorize(depth_to_level.get)
level_matrix = depth_to_level_vec(depth_matrix_full)

flood_df = pd.DataFrame(
    level_matrix,
    columns=[f"flood_level_t{i}" for i in range(level_matrix.shape[1])]
)

# Remove existing dynamic flood columns if they exist
edges_with_flood = edges_with_flood.loc[
    :,
    ~edges_with_flood.columns.str.startswith("flood_level_t")
]
edges_with_flood = pd.concat(
    [edges_with_flood.reset_index(drop=True), flood_df.reset_index(drop=True)],
    axis=1,
)

edges_with_flood.to_csv(
    "../data/processed/hatyai_edges_with_dynamic_flood.csv",
    index=False
)